In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Chapitre 1: Comprendre le machine learning

Considérons deux variables $y$ et $X$ données par les tableaux suivants:

$y = (y_1, y_2, \dots, y_n)$ et $x = (x_1, x_2, \dots, x_n)$. Notre but est d'établir une rélation entre les deux variables.


In [ ]:
data = pd.read_csv("1_regression_lineaire.csv")
data

In [ ]:
data.describe()

In [ ]:
# On va visualiser nos données 

plt.scatter(data['x'], data['y'])
plt.xlabel('x')
plt.ylabel('y')
plt.title('Regression linéaire 1')
plt.show()

## I - Par la méthode des moindres carrées ordinaires

En passant par la méthode des moindres carrés ordinaires on a :

$$y_{i} = \beta_{0} + \beta_{1}x_{i} + \epsilon_{i}$$

$$\epsilon_{i}^{2} = \left(y_{i} - (\beta_{0} + \beta_{1}x_{i})\right)^{2}$$

$$S(\beta_{0}, \beta_{1}) = \sum_{i=1}^{N}\epsilon_{i}^{2} = \sum_{i=1}^{N}\left(y_{i} - \beta_{0} - \beta_{1}x_{i}\right)^{2}$$

On cherche $(\hat\beta_{0}, \hat\beta_{1})$ qui minimisent $S$. On annule les dérivées partielles.

### Dérivée par rapport à $\beta_{0}$

$$\frac{\partial S}{\partial \beta_{0}} = -2\sum_{i=1}^{N}\left(y_{i} - \beta_{0} - \beta_{1}x_{i}\right) = 0$$

$$\sum_{i=1}^{N}y_{i} = N\beta_{0} + \beta_{1}\sum_{i=1}^{N}x_{i}$$

En divisant par $N$ (avec $\bar{x}$ et $\bar{y}$ les moyennes) :

$$\boxed{\hat\beta_{0} = \bar{y} - \hat\beta_{1}\bar{x}}$$

### Dérivée par rapport à $\beta_{1}$

$$\frac{\partial S}{\partial \beta_{1}} = -2\sum_{i=1}^{N}x_{i}\left(y_{i} - \beta_{0} - \beta_{1}x_{i}\right) = 0$$

$$\sum_{i=1}^{N}x_{i}y_{i} = \beta_{0}\sum_{i=1}^{N}x_{i} + \beta_{1}\sum_{i=1}^{N}x_{i}^{2}$$

On remplace $\beta_{0}$ par $\bar{y} - \beta_{1}\bar{x}$, avec $\sum x_{i} = N\bar{x}$ :

$$\sum_{i=1}^{N}x_{i}y_{i} = (\bar{y} - \beta_{1}\bar{x})N\bar{x} + \beta_{1}\sum_{i=1}^{N}x_{i}^{2}$$

$$\sum_{i=1}^{N}x_{i}y_{i} - N\bar{x}\bar{y} = \beta_{1}\left(\sum_{i=1}^{N}x_{i}^{2} - N\bar{x}^{2}\right)$$

Or $\sum x_{i}y_{i} - N\bar{x}\bar{y} = \sum (x_{i}-\bar{x})(y_{i}-\bar{y})$ et $\sum x_{i}^{2} - N\bar{x}^{2} = \sum (x_{i}-\bar{x})^{2}$, d'où :

$$\boxed{\hat\beta_{1} = \frac{\sum_{i=1}^{N}(x_{i}-\bar{x})(y_{i}-\bar{y})}{\sum_{i=1}^{N}(x_{i}-\bar{x})^{2}} = \frac{\mathrm{Cov}(x,y)}{\mathrm{Var}(x)}}$$

### Vérification que c'est bien un minimum

La matrice hessienne de $S$ est :

$$H = 2\begin{pmatrix} N & \sum x_{i} \\ \sum x_{i} & \sum x_{i}^{2} \end{pmatrix}$$

Son déterminant vaut :

$$\det H = 4\left(N\sum x_{i}^{2} - \left(\sum x_{i}\right)^{2}\right) = 4N\sum (x_{i}-\bar{x})^{2} > 0$$

et $2N > 0$, donc $H$ est définie positive : le point critique est un minimum (à condition que les $x_{i}$ ne soient pas tous égaux).

### Conclusion

La droite ajustée est :

$$\hat{y} = \hat\beta_{0} + \hat\beta_{1}x$$

Pour `1_regression_lineaire.csv`, on doit retrouver $\hat\beta_{1} \approx 3$ et $\hat\beta_{0} \approx 5$.

In [ ]:
data.describe()

In [ ]:
data.head(5)

$$\bar{x} = \frac{1}{N}\sum_{i=1}^{N}x_i$$

In [ ]:
x_bar = data['x'].mean()
y_bar = data['y'].mean()
# let's define cov and var functions
cov_xy = np.sum((data['x'] - x_bar) * (data['y'] - y_bar)) / len(data)
var_x = data['x'].var()

In [ ]:
beta_1 = cov_xy / var_x
beta_1

In [ ]:
beta_0 = y_bar - beta_1 * x_bar
beta_0

In [ ]:
plt.scatter(data['x'], data['y'], color='blue', label='Data points')
plt.plot(data['x'], beta_0 + beta_1 * data['x'], color='red', label='Regression line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

### II- Par la méthode de la descente de gradient

Vu que notre but est de déterminer le meilleur, il va falloir mésurer quand est-ce que le modèle se trompe, à travers une fonction de coût. La déduction, plus un modèle se trompe, plus ses paramètres sont mauvais.

- Considérons notre modèle de base:

$$ y_t = \beta_{0}+\beta_{1}X_{t} +\epsilon_{t}$$

- Définissons une fonction de coût:

$$J(w,b) = \frac{1}{2n}\sum_{i=1}^{n}(\hat{y_{t}}-y_{t})^{2}$$; avec $$\hat{y} = \hat{\beta_{0}} + \hat{\beta_{1}x}$$ et $\hat{\beta_{0}}, \hat{\beta_{1}}$ qui sont nos paramètres à initialiser.

- On va calculer le gradient de la fonction de coût:

$$\nabla J(\beta_{0},\beta_{1}) = \begin{pmatrix} \frac{\partial J(.)}{\partial \beta_0} \\ \frac{\partial J(.)}{\partial \beta_1} \end{pmatrix}$$

$$
    \begin{pmatrix} 
        \frac{1}{n}\sum_{i=1}^{n}(\beta_{0} + \beta_{1}X_{t} - y_{t}) \\
        \frac{1}{n}\sum_{i=1}^{n}(\beta_{0} + \beta_{1}X_{t} - y_{t})X_{t}
    \end{pmatrix}
$$

- A chaque étape on met à jour les paramètres dans le sens opposé du gradient dans la fonction de coût.

$$\beta_0 \longleftarrow \beta_0- \alpha \frac{\partial J}{\partial \beta_0}$$

$$\beta_1 \longleftarrow \beta_1-\alpha \frac{\partial J}{\partial \beta_1}$$

### Comment fonctionne l'algorithme de descente de gradient? 

- On initialise $\beta_{0}$ et $\beta_{1}$ avec des valeurs complètement arbitraires

- On calcule la fonction de coût, notre but est qu'elle soit proche de zéro. 

- Quand ce n'est pas le cas on met à jour les paramètres ou poids.

In [ ]:
from tqdm import tqdm
import random

# Paramètres initiaux et hyperparamètres
b0 = random.random()  # Biais (beta_0)
b1 = random.random()  # Poids (beta_1)
alpha = 0.01  # Taux d'apprentissage
epochs = 3000

n = len(data['x'])


def cost(b1, b0, X, y):
    total_cost = 0.0
    for i in range(len(X)):
        y_pred = b1 * X[i] + b0
        total_cost += (y_pred - y[i]) ** 2
    return total_cost / (2 * len(X))


# Liste pour stocker l'historique du coût si tu souhaites le tracer plus tard
cost_history = []

# Boucle d'entraînement avec tqdm
pbar = tqdm(range(1, epochs + 1), desc="Entraînement de la descente de gradient")

for epoch in pbar:
    beta_grad_0 = 0.0
    beta_grad_1 = 0.0

    # Calcul des gradients (accumulation)
    for i in range(n):
        y_pred = b1 * data['x'][i] + b0
        error = y_pred - data['y'][i]

        beta_grad_0 += error
        beta_grad_1 += error * data['x'][i]

    # Moyenne des gradients
    beta_grad_0 /= n
    beta_grad_1 /= n

    # Mise à jour des poids
    b0 = b0 - alpha * beta_grad_0
    b1 = b1 - alpha * beta_grad_1

    # Calcul du coût courant
    current_cost = cost(b1, b0, data['x'], data['y'])
    cost_history.append(current_cost)

    # Mise à jour dynamique des métriques dans la barre tqdm
    pbar.set_postfix({"Cost": f"{current_cost:.5f}", "b1": f"{b1:.3f}", "b0": f"{b0:.3f}"})

print("\n--- Entraînement terminé ---")
print(f"Biais final (b0)  : {b0:.4f}")
print(f"Poids final (b1)  : {b1:.4f}")
print(f"Coût final J(b1,b0): {current_cost:.6f}")